In [ ]:
!hf download malaysia-ai/Multilingual-TTS \
    --repo-type dataset \
    --local-dir "./data" \
    --exclude "*.zip"

In [1]:
import os 
assert os.environ.get("HF_TOKEN", None) is not None, "HF_TOKEN environment variable is not set. Please set it to your Hugging Face API token."

In [2]:
import pandas as pd
from datasets import load_dataset

/Users/tchiayan/Workspace/scicom/Multilingual-TTS/dnsmos/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_json("results_hf.jsonl", lines=True)

df["OVRL_3.2"] = df["OVRL"].apply(lambda x: x >= 3.2)
df_filtered = df[df["OVRL_3.2"]]
df_filtered.reset_index(drop=True, inplace=True)

In [4]:
df.groupby("OVRL_3.2")["filepath"].count()

OVRL_3.2
False    246393
True     413446
Name: filepath, dtype: int64

In [5]:
from functools import cache
@cache
def get_dataset(subset):
    try:
        ds = load_dataset("data", subset, split="train")
        dsdf = ds.to_pandas()
        dsdf.set_index("audio_filename", inplace=True)
        return dsdf
    except Exception as e:
        return None

from tqdm import tqdm

CORRECTION_MAP = {
    "IndicTTS_Punjabi":"IndicTTS",
}
def map_speaker_and_text(df): 
    rows = df.to_dict(orient="records")
    for row in tqdm(rows, total=len(rows)):
        subset = "_".join(row["filepath"].split("/")[0].split("_")[:-1])
        filename = "/".join(row["filepath"].split("/")[1:])
        row["filepath"] = filename
    
        
        if subset in CORRECTION_MAP:
            subset = CORRECTION_MAP[subset]
        # get the subset
        # continue
        dsdf = get_dataset(subset)
        row["subset"] = subset
        if dsdf is None or not filename in dsdf.index:
            # print(f"{filename} not found in {subset}")
            row["speaker"] = None
            row["text"] = None
            continue
        
        try:
            speaker = dsdf.loc[filename]["speaker"]
            text = dsdf.loc[filename]["text"]
            row["speaker"] = speaker
            row["text"] = text
        except Exception as e:
            print(dsdf)
            break
    
    df_mapped = pd.DataFrame(rows)
    df_mapped.dropna(subset=["speaker", "text"], inplace=True)
    df_mapped.reset_index(drop=True, inplace=True)
    return df_mapped
    
df_mapped = map_speaker_and_text(df_filtered)
df_mapped

100%|██████████| 413446/413446 [00:21<00:00, 19420.18it/s]


,filepath,text,speaker,len_in_sec,sr,num_hops,OVRL_raw,SIG_raw,BAK_raw,OVRL,SIG,BAK,OVRL_3.2,subset
0,700h-tr-turkish-text-to-speech_audio/gfs-01be5...,Hiç bu kadar büyük miktarda bağış almadıkların...,700h-tr-turkish-text-to-speech_audio_32,9.120000,16000,1,3.845674,4.144482,4.318467,4.200070,4.459289,4.507940,True,700h-tr-turkish-text-to-speech
1,700h-tr-turkish-text-to-speech_audio/gfs-01be5...,Türk taraflarlar paçavraları tırttı Fransız gü...,700h-tr-turkish-text-to-speech_audio_12,10.425375,16000,1,3.640357,3.968400,4.139748,3.995845,4.298047,4.345314,True,700h-tr-turkish-text-to-speech
2,700h-tr-turkish-text-to-speech_audio/gfs-01be5...,Donanmanın merkezi olan Gelibolu’da sahile yak...,700h-tr-turkish-text-to-speech_audio_34,9.705562,16000,1,3.737366,4.086297,4.149698,4.092913,4.406671,4.354679,True,700h-tr-turkish-text-to-speech
3,700h-tr-turkish-text-to-speech_audio/gfs-01be5...,Esenyurt Mehterçeşme Mahallesi’nde iki grup ar...,700h-tr-turkish-text-to-speech_audio_10,9.040000,16000,1,3.326967,3.617353,4.100873,3.675582,3.959551,4.308399,True,700h-tr-turkish-text-to-speech
4,700h-tr-turkish-text-to-speech_audio/gfs-01be5...,Aslında kırk yıl düşünsem annemden öğrendiğim ...,700h-tr-turkish-text-to-speech_audio_14,9.360000,16000,1,4.089436,4.363064,4.468325,4.436340,4.650851,4.634429,True,700h-tr-turkish-text-to-speech
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
326716,assamese_speech_dataset1_audio/assamese_speech...,আদালতত দৌৰি থকা খেলুৱৈ,assamese_speech_dataset1_audio_0,1.937437,16000,1,3.651414,4.248744,3.747562,4.006960,4.551885,3.951579,True,assamese_speech_dataset1
326717,assamese_speech_dataset1_audio/assamese_speech...,কিতাপ ষ্টেণ্ডত কিতাপৰ এটা গোট,assamese_speech_dataset1_audio_0,2.438125,16000,1,3.311203,3.904791,3.484511,3.659213,4.238350,3.666262,True,assamese_speech_dataset1
326718,assamese_speech_dataset1_audio/assamese_speech...,তিনিজন খেলুৱৈয়ে হেলমেট পিন্ধি আছে ।,assamese_speech_dataset1_audio_0,2.481688,16000,1,3.449825,3.966734,3.675092,3.802314,4.296492,3.874357,True,assamese_speech_dataset1
326719,assamese_speech_dataset1_audio/assamese_speech...,তুমি ব্যস্ত নেকি?,assamese_speech_dataset1_audio_0,1.458562,16000,1,3.479784,3.939218,3.712084,3.832991,4.270752,3.913920,True,assamese_speech_dataset1


In [6]:
df_mapped.drop(columns=["len_in_sec", "sr", "num_hops", "OVRL_3.2"], inplace=True)
df_mapped

,filepath,text,speaker,OVRL_raw,SIG_raw,BAK_raw,OVRL,SIG,BAK,subset
0,700h-tr-turkish-text-to-speech_audio/gfs-01be5...,Hiç bu kadar büyük miktarda bağış almadıkların...,700h-tr-turkish-text-to-speech_audio_32,3.845674,4.144482,4.318467,4.200070,4.459289,4.507940,700h-tr-turkish-text-to-speech
1,700h-tr-turkish-text-to-speech_audio/gfs-01be5...,Türk taraflarlar paçavraları tırttı Fransız gü...,700h-tr-turkish-text-to-speech_audio_12,3.640357,3.968400,4.139748,3.995845,4.298047,4.345314,700h-tr-turkish-text-to-speech
2,700h-tr-turkish-text-to-speech_audio/gfs-01be5...,Donanmanın merkezi olan Gelibolu’da sahile yak...,700h-tr-turkish-text-to-speech_audio_34,3.737366,4.086297,4.149698,4.092913,4.406671,4.354679,700h-tr-turkish-text-to-speech
3,700h-tr-turkish-text-to-speech_audio/gfs-01be5...,Esenyurt Mehterçeşme Mahallesi’nde iki grup ar...,700h-tr-turkish-text-to-speech_audio_10,3.326967,3.617353,4.100873,3.675582,3.959551,4.308399,700h-tr-turkish-text-to-speech
4,700h-tr-turkish-text-to-speech_audio/gfs-01be5...,Aslında kırk yıl düşünsem annemden öğrendiğim ...,700h-tr-turkish-text-to-speech_audio_14,4.089436,4.363064,4.468325,4.436340,4.650851,4.634429,700h-tr-turkish-text-to-speech
...,...,...,...,...,...,...,...,...,...,...
326716,assamese_speech_dataset1_audio/assamese_speech...,আদালতত দৌৰি থকা খেলুৱৈ,assamese_speech_dataset1_audio_0,3.651414,4.248744,3.747562,4.006960,4.551885,3.951579,assamese_speech_dataset1
326717,assamese_speech_dataset1_audio/assamese_speech...,কিতাপ ষ্টেণ্ডত কিতাপৰ এটা গোট,assamese_speech_dataset1_audio_0,3.311203,3.904791,3.484511,3.659213,4.238350,3.666262,assamese_speech_dataset1
326718,assamese_speech_dataset1_audio/assamese_speech...,তিনিজন খেলুৱৈয়ে হেলমেট পিন্ধি আছে ।,assamese_speech_dataset1_audio_0,3.449825,3.966734,3.675092,3.802314,4.296492,3.874357,assamese_speech_dataset1
326719,assamese_speech_dataset1_audio/assamese_speech...,তুমি ব্যস্ত নেকি?,assamese_speech_dataset1_audio_0,3.479784,3.939218,3.712084,3.832991,4.270752,3.913920,assamese_speech_dataset1


In [9]:
from datasets import Dataset, DatasetDict
ds_dict = DatasetDict({
      "_".join(subset_name.split("-")): Dataset.from_pandas(
          df_mapped[df_mapped["subset"] == subset_name].reset_index(drop=True)                                                                    
      ).rename_column("filepath", "audio_filename")
      for subset_name in df_mapped["subset"].unique()                                                                                             
  }) 
ds_dict

DatasetDict({
    700h_tr_turkish_text_to_speech: Dataset({
        features: ['audio_filename', 'text', 'speaker', 'OVRL_raw', 'SIG_raw', 'BAK_raw', 'OVRL', 'SIG', 'BAK', 'subset'],
        num_rows: 1965
    })
    9jalingo_hausa: Dataset({
        features: ['audio_filename', 'text', 'speaker', 'OVRL_raw', 'SIG_raw', 'BAK_raw', 'OVRL', 'SIG', 'BAK', 'subset'],
        num_rows: 57173
    })
    9jalingo_igbo: Dataset({
        features: ['audio_filename', 'text', 'speaker', 'OVRL_raw', 'SIG_raw', 'BAK_raw', 'OVRL', 'SIG', 'BAK', 'subset'],
        num_rows: 43863
    })
    9jalingo_pidgin: Dataset({
        features: ['audio_filename', 'text', 'speaker', 'OVRL_raw', 'SIG_raw', 'BAK_raw', 'OVRL', 'SIG', 'BAK', 'subset'],
        num_rows: 5452
    })
    Nanchang_Dialect_Conversational_Speech_Corpus: Dataset({
        features: ['audio_filename', 'text', 'speaker', 'OVRL_raw', 'SIG_raw', 'BAK_raw', 'OVRL', 'SIG', 'BAK', 'subset'],
        num_rows: 1195
    })
    CommonPhoneDataset

In [ ]:
ds_dict.push_to_hub(
    "tchiayan/Multilingual-TTS-DNSMOS", 
)

Setting num_proc from 1 back to 1 for the 700h_tr_turkish_text_to_speech split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 92.27ba/s]
Processing Files (1 / 1): 100%|██████████|  326kB /  326kB, 81.5kB/s  
New Data Upload: 100%|██████████|  326kB /  326kB, 81.5kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:05<00:00,  5.78s/ shards]
Setting num_proc from 1 back to 1 for the 9jalingo_hausa split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 17.09ba/s]
Processing Files (1 / 1): 100%|██████████| 4.57MB / 4.57MB, 1.52MB/s  
New Data Upload: 100%|██████████| 4.57MB / 4.57MB, 1.52MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:03<00:00,  3.88s/ shards]
Setting num_proc from 1 back to 1 for the 9jalingo_igbo split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow fo

CommitInfo(commit_url='https://huggingface.co/datasets/tchiayan/testing-dnsmos/commit/27c1ba31e1b11826294a68ba602689e1dcccc1eb', commit_message='Upload dataset', commit_description='', oid='27c1ba31e1b11826294a68ba602689e1dcccc1eb', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/tchiayan/testing-dnsmos', endpoint='https://huggingface.co', repo_type='dataset', repo_id='tchiayan/testing-dnsmos'), pr_revision=None, pr_num=None)